# TASK 1. PROJECT OVERVIEW & KEY LEARNING OBJECTIVES

# TASK 2. SETTING UP YOUR ENVIRONMENT

Before we can build our first AI agent, we need two things:

1. The `google-generativeai` library installed.  
2. Your free Gemini API key to communicate with Google's AI models.

> **Security tip 📌** Store keys securely in a `.env` file. Make sure you have a file named `.env` in the same directory as this notebook with your key:
>
> ```dotenv
> GEMINI_API_KEY=AIzaYourSecretGeminiKeyGoesHereXXXXXXXXX
> ```

Let's run the cells below to install the necessary packages and load your API key.

**You can get a free Gemini API key here: https://aistudio.google.com/app/apikey**

In [1]:
# Install required Python packages:
# google-generativeai: Google's official Python SDK for the Gemini API
# python-dotenv: loads environment variables from a .env file
%pip install google-generativeai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Let's import the necessary modules
import os
import google.generativeai as genai
from IPython.display import display, Markdown

# This will be used to load the API key from the .env file
from dotenv import load_dotenv
load_dotenv()

# Get the Gemini API key from environment variables
gemini_api_key = os.getenv("GEMINI_API_KEY")

# Configure the Gemini client using our key
genai.configure(api_key=gemini_api_key)
print("Gemini client successfully configured.")

# Let's view the first few characters in the key
print(gemini_api_key[:5])

c:\Users\EduTech\anaconda3\envs\py310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Gemini client successfully configured.
AIzaS


c:\Users\EduTech\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\hadoop\tmp\ipykernel_21392\1813551988.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
# A Function used to Show the given text using Markdown formatting in a Jupyter notebook
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

# TASK 3. BUILD & RUN YOUR FIRST AI AGENT USING GEMINI

It's time to build our first agent! We will use the `google-generativeai` library to create a simple AI agent backed by Gemini.

Creating an agent is like hiring an employee. You need to provide a **job description**. The key parameters are:

* **model_name** – The Gemini model to use as the AI brain (e.g., `gemini-1.5-flash` is fast, free-tier friendly, and capable).  
* **name** – A simple name to identify your agent (e.g., `"Fact Checker"`).  
* **instructions** – The most important part. It's the "system prompt" or the detailed job description for the agent. This tells the agent who to be and what to do.  

> **Free models available via Google AI Studio:**  
> - `gemini-1.5-flash` – fast, free tier available  
> - `gemini-1.5-pro` – more capable, free tier available  
> - `gemini-2.0-flash` – latest generation, free tier available  

In [4]:
# ---------------------------------------------------------------
# Agent class — a lightweight wrapper around the Gemini SDK
# that mimics the structure of the original OpenAI Agents SDK.
# ---------------------------------------------------------------
class Agent:
    """A simple AI agent backed by a Gemini model."""

    def __init__(self, name: str, instructions: str, model: str = "gemini-1.5-flash"):
        """
        name         – human-readable label for the agent
        instructions – system prompt that defines the agent's behaviour
        model        – Gemini model string (default: gemini-1.5-flash)
        """
        self.name = name
        self.instructions = instructions
        self.model = model
        # Create the underlying Gemini generative model with a system instruction
        self._client = genai.GenerativeModel(
            model_name=model,
            system_instruction=instructions,
        )


# ---------------------------------------------------------------
# Runner class — executes the agent on a given input and
# returns a response object with a .final_output attribute.
# ---------------------------------------------------------------
class RunResult:
    """Holds the result of a Runner.run() call."""

    def __init__(self, final_output: str, usage=None):
        self.final_output = final_output
        self.usage = usage  # token usage metadata (if available)


class Runner:
    """Runs an Agent against a user input and returns a RunResult."""

    @staticmethod
    def run(starting_agent: Agent, input: str) -> RunResult:
        """
        starting_agent – the Agent to use
        input          – the user message / statement to process
        """
        response = starting_agent._client.generate_content(input)
        return RunResult(
            final_output=response.text,
            usage=response.usage_metadata,
        )


# ---------------------------------------------------------------
# Define the instructions for the fact-checker AI Agent
# ---------------------------------------------------------------
fact_checker_instructions = """
Context:
You are a fact-checker who verifies the accuracy of statements.

Instructions:
When given a statement, carefully analyze its factual accuracy using your knowledge.

Input:
You will receive a statement that requires fact-checking.

Output:
Respond with:
1. A verdict prefix: either "✅ TRUE:" or "❌ FALSE:"
2. A brief, one-sentence explanation justifying your conclusion
"""

In [5]:
# Create a new agent called "Fact Checker"
fact_checker_agent = Agent(
    name="Fact Checker",               # Name of the agent
    instructions=fact_checker_instructions,  # The rules and behaviour for the agent
    model="gemini-2.5-flash-lite",          # Free-tier Gemini model
)

# Print a confirmation message that the agent was created
print(f"Agent '{fact_checker_agent.name}' created successfully!")

Agent 'Fact Checker' created successfully!


Let's put our `fact_checker_agent` to work! We'll use the `Runner.run()` method to send it a statement to fact-check. The agent will analyse the statement and return its verdict along with a brief explanation.

In [6]:
# A statement we want the Fact Checker agent to verify
statement = "The Great Wall of China is visible from space with the naked eye."

# Display the statement we're going to check (in markdown format for nicer formatting)
print_markdown(f"Asking the Fact Checker to verify: '{statement}'")

# Run the Fact Checker agent on the input statement
# NOTE: Unlike the OpenAI Agents SDK, Runner.run() here is synchronous —
# no 'await' keyword is needed.
response = Runner.run(
    starting_agent=fact_checker_agent,  # The agent we created earlier
    input=statement                     # The statement we want it to fact-check
)

# Display the agent's response
print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)    # Shows the final verdict and explanation

Asking the Fact Checker to verify: 'The Great Wall of China is visible from space with the naked eye.'


🤖 Agent's Response:


❌ FALSE: While the Great Wall of China is an immense structure, it is too narrow to be seen from space with the naked eye.

**PRACTICE OPPORTUNITY:** 
- **Now it's your turn to experiment with the Gemini AI Agent; perform the following tasks:**
   - **Change the text inside the `statement` variable. Try a different fact, like `"The tallest mountain in the world is Mount Everst"`. See how the AI agent responds!**
   - **Try a different Gemini model — change `model="gemini-1.5-flash"` to `model="gemini-1.5-pro"` or `model="gemini-2.0-flash"`**

## TASK 4. CHECK IF THE AI AGENT CAN RECALL INFORMATION (NO MEMORY)

In [7]:
# Ask the agent about the previous conversation — it should have NO memory
check_recall_statement = "What did we discuss in the last message?"

response = Runner.run(
    starting_agent=fact_checker_agent,
    input=check_recall_statement
)
print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)


🤖 Agent's Response:


❌ FALSE: I cannot access previous conversation history, so I do not know what we discussed in the last message.

- **Note: Each call to `Runner.run()` is stateless — the agent starts fresh every time with no memory of previous interactions.**
- **You can inspect token usage (input/output tokens) via `response.usage`:**

In [8]:
# Show token usage for the last response
# Gemini returns usage_metadata with prompt_token_count and candidates_token_count
print("Token Usage:")
print(f"  Input  tokens : {response.usage.prompt_token_count}")
print(f"  Output tokens : {response.usage.candidates_token_count}")
print(f"  Total  tokens : {response.usage.total_token_count}")

Token Usage:
  Input  tokens : 98
  Output tokens : 23
  Total  tokens : 121


# PRACTICE OPPORTUNITY SOLUTIONS

**PRACTICE OPPORTUNITY SOLUTION:**
- Changing the `statement` variable to test a new fact:
- Switching to `model="gemini-1.5-pro"` for a more capable model

In [9]:
# Create a new agent using a more capable free model
fact_checker_agent_pro = Agent(
    name="Fact Checker",
    instructions=fact_checker_instructions,
    model="gemini-2.5-pro",   # More capable free-tier model
)

print(f"Agent '{fact_checker_agent_pro.name}' created successfully!")

Agent 'Fact Checker' created successfully!


In [10]:
# Test with a new statement (note the deliberate typo in "Everst")
statement = "The tallest mountain in the world is Mount Everst"

print_markdown(f"Asking the Fact Checker to verify: '{statement}'")

response = Runner.run(
    starting_agent=fact_checker_agent_pro,
    input=statement
)

print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)

Asking the Fact Checker to verify: 'The tallest mountain in the world is Mount Everst'

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
Please retry in 59.751598547s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "gener

**PRACTICE OPPORTUNITY SOLUTION — Tweet Bot:**
- **Create a new AI agent named `Tweet Bot` that writes a creative, engaging tweet (≤280 characters) about a given topic using the Gemini SDK. The tweet should include at least one emoji and a hashtag.**

In [11]:
# ---------------------------------------------------------------
# 1. Define instructions for the Tweet Bot agent
# ---------------------------------------------------------------
tweet_bot_instructions = """
You are Tweet Bot, a creative and engaging social media expert.

Instructions:
When given a topic, write a single tweet that:
  - Is no longer than 280 characters
  - Includes at least one relevant emoji
  - Includes at least one relevant hashtag
  - Is engaging, positive, and thought-provoking

Output ONLY the tweet text. No additional commentary.
"""

# ---------------------------------------------------------------
# 2. Create the Tweet Bot agent
# ---------------------------------------------------------------
tweet_bot_agent = Agent(
    name="Tweet Bot",
    instructions=tweet_bot_instructions,
    model="gemini-2.5-flash-lite",  # Fast, free-tier model
)

print(f"Agent '{tweet_bot_agent.name}' created successfully!")

# ---------------------------------------------------------------
# 3. Choose a topic and run the agent
# ---------------------------------------------------------------
topic = "IRAN-US WAR"  # You can change this topic to test with different inputs

response = Runner.run(
    starting_agent=tweet_bot_agent,
    input=topic
)

tweet = response.final_output.strip()

print_markdown(f"**Topic:** {topic}")
print_markdown("\n🐦 Generated Tweet:\n")
print_markdown(f"> {tweet}")
print(f"\nCharacter count: {len(tweet)} / 280")

# Show token usage
print("\nToken Usage:")
print(f"  Input  tokens : {response.usage.prompt_token_count}")
print(f"  Output tokens : {response.usage.candidates_token_count}")
print(f"  Total  tokens : {response.usage.total_token_count}")

Agent 'Tweet Bot' created successfully!


**Topic:** IRAN-US WAR


🐦 Generated Tweet:


> While tensions are high, let's focus on de-escalation and finding peaceful solutions. Diplomacy is key to navigating complex international relations. 🕊️ #PeaceNotWar


Character count: 165 / 280

Token Usage:
  Input  tokens : 89
  Output tokens : 35
  Total  tokens : 124
